In [1]:
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
import numpy as np
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_OFFLINE"] = "1" #离线加载

In [2]:
with open('习近平党建数据库-3.txt', 'r', encoding='utf-8') as file:
  sentences = file.readlines()
  # sentences = file.readlines()[:100] # ⚠️⚠️⚠️测试的时候可以只看前100条数据⚠️⚠️⚠️
print('文本条数: ', len(sentences))
print('预览第一条: ', sentences[0])

文本条数:  6589
预览第一条:  二零一二年。



In [3]:
model_name = "bert-base-chinese"
# embedding_model=SentenceTransformer("C:\\Users\\xule\\.cache\\huggingface\\hub\\models--bert-base-chinese\\snapshots\\c30a6ed22ab4564dc1e3b2ecbf6e766b0611a33f")
model = BertModel.from_pretrained(model_name)
tokenizer = BertTokenizer.from_pretrained(model_name)

In [4]:
# 将模型放置在GPU上
# torch.cuda.is_available()，检测cuda是否可用
# torch.device()设置张量运算在哪个设备上进行
# device = torch.device("cpu")，表示在CPU进行
# device = torch.device("cuda")，表示在GPU进行
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 把模型放到cpu或gpu
model.to(device)
# 将模型设置为评估模式，https://blog.csdn.net/weixin_45275599/article/details/131524189
model.eval()

# 切分数据
batch_size = 16  # 批大小
data_loader = DataLoader(sentences, batch_size=batch_size)
for batch in data_loader:
    print(len(batch), batch)

16 ['二零一二年。\n', '中央八项规定体现党要管党、从严治党，是一个切入口和动员令\n', '新一届中央领导集体要定规矩〔1,这是很重要的规矩。\n', '没有规矩，不成方圆。\n', '从我们在座各位做起来，新人新办法。\n', '制定这方面的规矩，指导思想就是从严要求，体现党要管党、从严治党。\n', '党风廉政建设，要从领导干部做起，领导干部首先要从中央领导做起。\n', '正所谓己不正，焉能正人。\n', '最重要的就是要防微杜渐，不要“温水煮青蛙”。\n', '现在，有些形式主义、官僚主义的东西，有些铺张浪费、豪华奢侈的东西，上上下下都有些表现，我们不能安之若素、司空见惯、见怪不怪。\n', '既然作规定，就要朝严一点的标准去努力，就要来真格的。\n', '不痛不痒的，四平八稳的，都是空洞口号，就落不到实处，还不如不做。\n', '定规矩，就要落实一些已经有明确规范的事情，就要约束一些不合规范的事情，就要规范一些没有规范的事情。\n', '规矩是起约束作用的，所以要紧一点。\n', '紧一点自然就不舒服了，舒适度就有问题了，就是要不舒服一点、不自在一点，我们不舒服一点、不自在一点，老百姓的舒适度就好一点、满意度就高一点，对我们的感觉就好一点。\n', '这也是新形象新气象。\n']
16 ['规定就是规定，不加“试行”两字，就是要表明一个坚决的态度，表明这个规定是刚性的。\n', '“试行”给人感觉是不是还有点含糊。\n', '就先按这个规定去做，做了以后真正推开了，一两年后再完善。\n', '中央纪委有那么多规定，不也就规定下去了。\n', '反正要不断去约束。\n', '最重要的是要抓好落实，言必行、行必果。\n', '我们说了不是白说，说了就必须做到，把文件上写的内容一一落到实处。\n', '要完善细则，警卫、新闻、文秘、内事、外事都要有各自的细化落实方案。\n', '规定制定出来后，大家都要学习贯彻，特别是领导干部身边的工作人员要学习贯彻。\n', '改进工作作风的任务非常繁重，八项规定是一个切入口和动员令。\n', '八项规定既不是最高标准，更不是最终目的，只是我们改进作风的第一步，是我们作为共产党人应该做到的基本要求。\n', '“善禁者，先禁其身而后人。\n', '”〔2〕各级领导干部要以身作则、率先垂范，说到的就要做到，承诺的就要兑

In [5]:
# ---- 文本转向量 ----
# 生成的向量存放在这里
cls_embeddings = []

# 使用tqdm显示处理进度
# tqdm b站教程：https://www.bilibili.com/video/BV1ZG411M7Ge/?spm_id_from=333.337.search-card.all.click&vd_source=eace37b0970f8d3d597d32f39dec89d8
for batch_sentences in tqdm(data_loader):
    # tokenizer官方文档：https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.__call__
    # truncation=True，对输入句子进行截断，这里确保最大长度不超过512个字
    # max_length：不设置的话，默认会截断到该模型可接受的最大长度
    # padding=True 或 padding='longest': 将所有句子填充到批次中最长句子的长度
    # padding="max_length": 将所有句子填充到由 max_length 参数指定的长度
    inputs = tokenizer(batch_sentences, padding=True, truncation=True, return_tensors="pt", max_length=512)
    # print(123, inputs.input_ids[0], tokenizer.decode(inputs.input_ids[0]))
    
    # 把编码好的数据，也放在device上，It is necessary to have both the model, and the data on the same device, either CPU or GPU
    # https://huggingface.co/docs/transformers/v4.39.2/en/main_classes/tokenizer#transformers.BatchEncoding.to
    # https://stackoverflow.com/questions/63061779/pytorch-when-do-i-need-to-use-todevice-on-a-model-or-tensor
    inputs.to(device)

    # 设置不要计算梯度
    # 一般来说，如果我们只是用模型进行“预测”，而不涉及对模型进行更新时，就不需要计算梯度，以此来节约内存，增加运算效率
    # with上下文中，对model的调用将遵循torch.no_grad()，即不会计算梯度
    with torch.no_grad():
        outputs = model(**inputs)

    # 把这一批词向量存入cls_embeddings容器中
    # tensor.cpu() 将张量移动到 CPU
    # tensor.numpy() 将 CPU 上的张量转换为 NumPy 数组
    cls_embeddings.append(outputs.last_hidden_state[:, 0].cpu().numpy()) # 只取CLS对应的向量

    # print('pt格式', type(outputs.last_hidden_state[:, 0].shape), outputs.last_hidden_state[:, 0].shape)
    print('numpy格式', type(outputs.last_hidden_state[:, 0].cpu().numpy()), outputs.last_hidden_state[:, 0].cpu().numpy().shape)

# 合并句子向量
print('batch个数：', len(cls_embeddings))
cls_embeddings_np = np.vstack(cls_embeddings)
print('最终生成的词向量', type(cls_embeddings_np), cls_embeddings_np.shape)

# ---- 保存词嵌入向量 ----
# 保存句子向量到npy文件
# 官方文档：https://numpy.org/doc/stable/reference/generated/numpy.save.html
output_file = "emb_习近平党建数据库.npy"
np.save(output_file, cls_embeddings_np)
print("词向量存储于: ", output_file)

embeddings = np.load(output_file)
print("加载回来，验证一下：", type(embeddings), embeddings.shape)

  1%|          | 4/412 [00:00<00:36, 11.14it/s]

numpy格式 <class 'numpy.ndarray'> (16, 768)
numpy格式 <class 'numpy.ndarray'> (16, 768)
numpy格式 <class 'numpy.ndarray'> (16, 768)
numpy格式 <class 'numpy.ndarray'> (16, 768)


  2%|▏         | 8/412 [00:00<00:30, 13.44it/s]

numpy格式 <class 'numpy.ndarray'> (16, 768)
numpy格式 <class 'numpy.ndarray'> (16, 768)
numpy格式 <class 'numpy.ndarray'> (16, 768)
numpy格式 <class 'numpy.ndarray'> (16, 768)


  2%|▏         | 10/412 [00:00<00:27, 14.75it/s]

...

numpy格式 <class 'numpy.ndarray'> (16, 768)
numpy格式 <class 'numpy.ndarray'> (16, 768)
numpy格式 <class 'numpy.ndarray'> (16, 768)


 99%|█████████▉| 408/412 [00:26<00:00, 14.80it/s]

numpy格式 <class 'numpy.ndarray'> (16, 768)
numpy格式 <class 'numpy.ndarray'> (16, 768)
numpy格式 <class 'numpy.ndarray'> (16, 768)
numpy格式 <class 'numpy.ndarray'> (16, 768)


100%|██████████| 412/412 [00:26<00:00, 15.32it/s]

numpy格式 <class 'numpy.ndarray'> (16, 768)
numpy格式 <class 'numpy.ndarray'> (16, 768)
numpy格式 <class 'numpy.ndarray'> (13, 768)
batch个数： 412
最终生成的词向量 <class 'numpy.ndarray'> (6589, 768)
词向量存储于:  emb_习近平党建数据库.npy
加载回来，验证一下： <class 'numpy.ndarray'> (6589, 768)
